# A multi-factor market-neutral investment strategy for New York Stock Exchange equities

This notebook implements the strategy described in the paper by Georgios M. Gkolemis, Adwin Richie Lee, and Amine Roudani (2024). The strategy aims to deliver steady returns while minimizing correlation with the market by using a combination of momentum-based indicators, fundamental factors, and analyst recommendations.

**Paper citation:**
Gkolemis, G. M., Lee, A. R., & Roudani, A. (2024). A multi-factor market-neutral investment strategy for New York Stock Exchange equities. *arXiv preprint arXiv:2412.12350*.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT']
START_DATE = '2020-01-01'
END_DATE = '2023-12-31'
RISK_FREE_RATE = 0.02

# Hypothesis
# The strategy hypothesizes that a combination of momentum, fundamental factors, and analyst recommendations can identify stocks that will outperform or underperform the market, allowing for a market-neutral portfolio with steady returns.


## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start=START_DATE, end=END_DATE, group_by='ticker')

# Compute features
def compute_features(data):
    features = {}
    for ticker, df in data.items():
        df['Return'] = df['Adj Close'].pct_change()
        df['Momentum'] = df['Return'].rolling(window=12).mean()
        df['Volatility'] = df['Return'].rolling(window=60).std()
        features[ticker] = df[['Return', 'Momentum', 'Volatility']]
    return features

features = compute_features(data)

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Signal generation
def generate_signals(features):
    signals = {}
    for ticker, df in features.items():
        df['Signal'] = np.where(df['Momentum'] > 0, 1, -1)
        signals[ticker] = df['Signal']
    return signals

signals = generate_signals(features)

# Portfolio construction
def construct_portfolio(signals):
    portfolio = pd.DataFrame(index=signals[ticker].index)
    for ticker, signal in signals.items():
        portfolio[ticker] = signal
    portfolio = portfolio.fillna(0)
    return portfolio

portfolio = construct_portfolio(signals)

## Phase 4 — Vectorized Backtest

In [ ]:
# Backtest
def backtest(portfolio, features):
    returns = pd.DataFrame(index=portfolio.index)
    for ticker in portfolio.columns:
        returns[ticker] = features[ticker]['Return'] * portfolio[ticker].shift(1)
    returns['Portfolio'] = returns.sum(axis=1)
    returns['Cumulative'] = (1 + returns['Portfolio']).cumprod()
    return returns

backtest_results = backtest(portfolio, features)

## Phase 5 — Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Performance metrics
def calculate_metrics(returns):
    excess_returns = returns['Portfolio'] - RISK_FREE_RATE
    sharpe = np.mean(excess_returns) / np.std(excess_returns)
    sortino = np.mean(excess_returns) / np.std(np.where(excess_returns < 0, excess_returns, 0))
    calmar = sharpe / (returns['Cumulative'].max() / returns['Cumulative'].min())
    max_drawdown = (returns['Cumulative'].cummax() - returns['Cumulative']).max()
    
    print(f'Sharpe Ratio: {sharpe:.2f}')
    print(f'Sortino Ratio: {sortino:.2f}')
    print(f'Calmar Ratio: {calmar:.2f}')
    print(f'Max Drawdown: {max_drawdown:.2%}')
    
    # Plot equity curve
    plt.figure(figsize=(10, 5))
    plt.plot(returns['Cumulative'], label='Equity Curve')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Returns')
    plt.title('Equity Curve')
    plt.legend()
    plt.show()

calculate_metrics(backtest_results)

## Phase 6 — Monitoring Stub

In [ ]:
# Monitoring stub
def monitor(portfolio, features):
    current_date = features[ticker].index[-1]
    current_positions = portfolio.iloc[-1]
    current_returns = features[ticker]['Return'].iloc[-1] * current_positions
    daily_pnl = current_returns.sum()
    
    print(f'Date: {current_date}')
    print(f'Daily P&L: {daily_pnl:.2f}')
    print('Current Positions:')
    print(current_positions)

monitor(portfolio, features)